# LSTM Testing
---

Our dataset in this project is an open-source gas dataset hosted on the [UCI Machine Learning Repository Website](https://archive.ics.uci.edu/dataset/360/air+quality). It contains time series data on different gas concentrations on a field in an Italian city, which we can train our LSTM model on. If you fork this repo, you should have the dataset preinstalled. For this project, we will focus only on the concentration of carbon monoxide.

## Loading and Prepping Data
We'll begin by loading the data from our dataset using the pandas library. Before we do any sort of training, we need to prep our data for training.

In [ ]:
import pandas as pd

# Load the dataset
df = pd.read_csv("AirQualityUCI.csv", delimiter=";")

# Print the data that we have to see what our data looks like. Alternatively, you can open the data in Excel or any other viewer
print(df.head)

Interesting to note that we have NaN values. We'll want to remove those from our dataset.

In [ ]:
# Pandas builtin method to remove Nonetype or NaN data
df = df.dropna(subset=['CO(GT)'])

# Print CO gas concentrations with custom headers.
print(f"ID\tCO Concentration")
print(df['CO(GT)'].sort_values())

This is better, but notice that we seem to have some values of -200, which does not make sense since we are measuring the concentration of CO. A negative value should not exist, thus we need to remove that as well. At the same time, we'll convert the column datatype from string to float:

In [ ]:
df['CO(GT)'] = (
    df['CO(GT)']
    .replace(regex=',',value='.')   # replace , with . so that numpy can convert to float...
    .astype(float)                  # ...which happens right here!
)

# Now that our datatype is numerical, we can filter out negative values
df = df[(df['CO(GT)'] > 0)]

# Print out filtered data with custom headers
print(f"ID\tCO Concentration")
print(df['CO(GT)'])

Now we are almost ready to plot the data. Since our data is taken over a timespan, we will want our x-axis to be time. Notice that time data is split between two columns: 
- Date
- Time

For each day, 24 measurements were taken. We will have to merge these columns so that we can have a continuous line of time data.

In [ ]:
# We first replace the time separator . with :, and merge the Date and Time columns into a new Datetime column
df['Datetime'] = df['Date'] + ' ' + df['Time'].replace(regex=r"\.",value=":")

# Next we convert the datatype from string to datetime
df['Datetime'] = pd.to_datetime(df['Datetime'],format='%d/%m/%Y %H:%M:%S')

# Print out datetime values with custom headers
print(f"ID\tDatetime")
print(df['Datetime'])

## Plotting Our Data
Now we can finally plot our graph, and see what our data looks like.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Define x and y axes
x = df['Datetime'].to_numpy()
y = df['CO(GT)']

# Formatting etc.
fig, ax = plt.subplots()
ax.yaxis.set_major_locator(plt.MultipleLocator(0.5))
plt.plot(x,y)
plt.title("CO Levels Over Time")
plt.ylabel("CO Concentration (mg/m^3)")
plt.xlabel("Time")
plt.tick_params(axis='y',pad=10)

plt.show()

## Preprocessing

We are almost done. In order to be used for training in an LSTM model, we need two things from this data:
1. Normalized. We will use a minmax function to normalize the concentrations to values between 0 and 1
2. Windows. An LSTM is used for processing timeseries data. For an LSTM to work, it requires past information to predict the future. We do this by creating a sliding window.

In [ ]:
# Normalize the CO concentration values
series = df['CO(GT)'].values
norm_series = np.array((series - np.min(series)) / (np.max(series) - np.min(series)))

# Once again plotting the data
x = df['Datetime']
y = norm_series

fig, ax = plt.subplots()
ax.yaxis.set_major_locator(plt.MultipleLocator(0.05))
plt.plot(x,y)
plt.title("CO Levels Over Time (normalized)")
plt.ylabel("CO Concentration")
plt.xlabel("Time")
plt.tick_params(axis='y',pad=10)
plt.show()


Note that the graph looks exactly the same, but now the values have been squished to a range between 0 and 1. Following our notes on [sliding windows](README.md#the-sliding-window), we can now construct our sliding windows, and divide them into our 80:20 train-test split.

In [ ]:
# We give the model the last 24 hours' worth of information to predict the future
window_size = 24

X, y = [], []

# Here we create our sliding windows
for i in range(len(norm_series) - window_size):
        X.append(norm_series[i : i + window_size])
        y.append(norm_series[i + window_size])

# Splitting the data into testing and training values
np.random.shuffle(X)
np.random.shuffle(y)
training_X, testing_X = X[:len(X)*3//4], X[len(X)*3//4:]
training_y, testing_y = X[:len(X)*3//4], X[len(X)*3//4:]

"""
TODO: Figure out what this is doing and why we do it.
"""
# Dimensionality so that it can be used as a tensor
training_X = np.expand_dims(training_X, axis=-1)
training_y = np.expand_dims(training_y, axis=-1)
testing_X = np.expand_dims(testing_X, axis=-1)
testing_y = np.expand_dims(testing_y, axis=-1)

print(f"Size of total dataset:\t\t|\t{len(df['CO(GT)'])} elements")
print(f"Size of training dataset:\t|\t{len(training_X)} elements")
print(f"Size of testing dataset:\t|\t{len(testing_X)} elements")


## Training the LSTM model
It is finally time to start the training process. We define a sequential model with the following structure:

Input (24 values with one feature) -> LSTM layer (with 64 neurons) -> LSTM layer (with 64 neurons) -> Output layer (1 neuron)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import ReduceLROnPlateau

# Create a rlop callback function to be able to more accurately get to the global minima
rlrop = ReduceLROnPlateau()

# We only do 20 epochs in this example to save some time, and as you will see, the model approaches the minima very quickly
EPOCHS = 20

# In order for us to get some metrics on how the training process is going, we save 5% of the training data to evaluate on as we train
VAL_SPLIT = 0.05

# Defining the model
model = Sequential([
    tf.keras.layers.Input(shape=(window_size,1)),
    LSTM(64,return_sequences=True),
    LSTM(64,return_sequences=False),
    Dense(1)
])

model.compile(loss ='mse', optimizer='adam')

# Begin the training. We save metric data as a history object
history = model.fit(
    x=training_X,
    y=training_y,
    batch_size=32,
    epochs=EPOCHS,
    validation_split=VAL_SPLIT,
    callbacks=[rlrop]
)

# Plot Training vs Validation Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss Evolution')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()



Beautiful! Our model has been trained, and now it is time to evaluate it with our other split of data we created earlier.

## Evaluating the Model

In [ ]:
result = model.evaluate(
    x=testing_X,
    y=testing_y,
    batch_size=32
)

print(f"Testing loss: {result}")


The model has now been trained! Notice that we know that the model was not overfitting, as we showed the model data it had never seen before, and it was able to predict the next timestep fairly accurately. The model is now done.

### Extra reading:
- [Some notes on LSTMs](README.md#notes-on-lstms)